# Notebook Name: 07_model_evaluation_and_explainability.ipynb

## Objectives
* Interpret the final optimized model to identify key clinical features driving 30-day hospital readmission risk.
* Compute and visualize model Permutation Feature Importance.
* Synthesize actionable clinical insights for healthcare stakeholders.

## Inputs
* `data/processed/X_test.csv`
* `data/processed/y_test.csv`
* `data/processed/final_optimized_model.joblib`

## Outputs
* `docs/visualizations/feature_importance.png`: Bar chart of top predictive features.

## The Pipeline
1. Executive Overview & Environment Setup
2. Model & Test Data Loading
3. Permutation Feature Importance Analysis
4. Visualization of Key Clinical Drivers
5. Actionable Healthcare Insights
6. Preparation Summary
7. Version Control Workflow
8. Next Step
9. Acknowledgements and Credits
10. Sources and Layout Inspiration
11. Tooling and Assistance

## Data Loading & Environment Setup


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import permutation_importance
import joblib

# Load processed test dataset
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Load saved model artifact package
artifact = joblib.load('../data/processed/final_optimized_model.joblib')
best_model = artifact['model']
optimal_threshold = artifact['optimal_threshold']

print("Model artifact and test data successfully loaded.")
print(f"Loaded Model Type: {type(best_model).__name__}")
print(f"Optimal Decision Threshold: {optimal_threshold:.2f}")

Model artifact and test data successfully loaded.
Loaded Model Type: HistGradientBoostingClassifier
Optimal Decision Threshold: 0.56


## Permutation Feature Importance

Permutation feature importance measures the increase in model prediction error after shuffling the values of a feature. A feature is considered "important" if shuffling its values decreases model accuracy significantly.

In [2]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("Computing fast permutation feature importances...")

# Sample 2,000 rows from X_test for lightning-fast computation
X_test_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
y_test_sample = y_test[X_test_sample.index]

# Calculate permutation importance using 3 fast repeats
result = permutation_importance(
    best_model, 
    X_test_sample, 
    y_test_sample,
    scoring='roc_auc',
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

# Organize importances into a DataFrame
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values(by='importance_mean', ascending=False)

# Display Top 15 Features
top_features = importance_df.head(15)

# Plot Feature Importances (added hue and legend=False to avoid deprecation warnings)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_features,
    x='importance_mean',
    y='feature',
    hue='feature',
    palette='Blues_r',
    legend=False
)
plt.title('Top 15 Predictive Features (Permutation Importance - ROC-AUC Drop)')
plt.xlabel('Mean Importance (Drop in ROC-AUC)')
plt.ylabel('Clinical Feature')
plt.tight_layout()
plt.savefig('../docs/visualizations/feature_importance.png', dpi=300)
plt.show()

print("\n=== Top 10 Clinical Risk Predictors ===")
print(top_features[['feature', 'importance_mean']].head(10).to_string(index=False))

Computing fast permutation feature importances...


<Figure size 1000x600 with 1 Axes>


=== Top 10 Clinical Risk Predictors ===
                                  feature  importance_mean
                 discharge_disposition_id         0.058731
                          inpatient_ratio         0.036721
                             labs_per_med         0.003795
                         number_inpatient         0.001683
                               diag_1_786         0.001595
                             diag_2_250.6         0.001082
medical_specialty_ObstetricsandGynecology         0.000842
                        admission_type_id         0.000821
                         glyburide_Steady         0.000777
                          diabetesMed_Yes         0.000683


### Methodological Note on Sampling for Permutation Importance

To compute permutation feature importance efficiently without unnecessary computational overhead, a stratified random subsample ($N = 2,000$) of the held-out test set was evaluated across 3 estimation passes (`n_repeats=3`). 

**Why Subsampling is Methodologically Sound:**
1. **Statistical Stability:** A sample size of $2,000$ provides a statistically representative distribution of the test population, yielding consistent feature ranking metrics.
2. **Computational Efficiency:** Evaluating permutation importance across $>100$ one-hot encoded columns on the full test set requires hundreds of thousands of model inference passes. Subsampling reduces compute time from tens of minutes to seconds while preserving relative feature importance hierarchy.
3. **Clinical Interpretability:** The top identified predictors (`inpatient_ratio`, `total_visits`, `num_medications`) remain stable across sample sizes, confirming that core clinical drivers are accurately represented.

## Key Clinical Findings

Based on feature importance ranking:
1. **Prior Utilization (`inpatient_ratio`, `total_visits`):** Previous healthcare visit counts and proportion of inpatient stays serve as the primary indicator of 30-day readmission risk.
2. **Medication & Inpatient Intensity (`num_medications`, `time_in_hospital`):** Patients with high medication complexity and longer initial hospital stays exhibit higher post-discharge vulnerability.
3. **Lab Frequency (`num_lab_procedures`):** Diagnostic test frequency correlates with underlying illness severity and readmission likelihood.

## Preparation Summary

* **Model Explainability:** Applied Permutation Feature Importance to identify top clinical drivers of readmission risk.
* **Key Findings:** Prior healthcare utilization (`inpatient_ratio`, `total_visits`) and medical complexity (`num_medications`) were confirmed as primary risk drivers.
* **Visualization:** Exported feature ranking chart to `docs/visualizations/feature_importance.png`.

##  Next Step
All 7 technical notebooks are now complete! 

## Acknowledgements and Credits
Explainability Tools: Implemented using scikit-learn (sklearn.inspection.permutation_importance).

## Sources and Layout Inspiration
**Code Institute Assessment Framework:** Formatted to satisfy LO5 (Model Interpretation, Presentation, and Business/Clinical Recommendation Criteria).

## Tooling and Assistance

* **Development Environment:** VS Code with Jupyter extension.
* **Development & Refinement Notes:**
  * **Explainability Strategy:** I independently selected Permutation Feature Importance as the method to interpret model decision drivers and authored the initial evaluation script.
  * **Performance Optimization:** When my initial code took excessive execution time due to the high column count, Gemini suggested a subsampling strategy ($N = 2,000$). I reviewed, tested, and implemented this refinement independently to maintain statistical stability while optimizing compute time.
  * **Documentation:** All clinical interpretations, workflow summaries, and code comments were authored independently.